# 🔘🖼️ Gradio SDXL Image-to-Image

⚠️ **Remember to copy this notebook in your Drive and rename.**

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

**🗒️ Gradio's [Documentation](https://www.gradio.app/docs)**

Gradio's [Documentation](https://www.gradio.app/docs)

## Install Gradio (and Restart)

In [ ]:
!pip install gradio --quiet

### Mount Drive

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Setup

In [ ]:
%cd /content
!rm -rf iaac_genai
!git clone https://github.com/jamesmcbennett/iaac_genai
%cd /content/iaac_genai/

In [ ]:
import sys
sys.path.append('/content/iaac_genai')

In [ ]:
!pip install -q -r requirements.txt --quiet > /dev/null 2>&1

In [ ]:
from config import Config
from utils import set_image_path, save_image, save_yml, save_svg
import torch

from diffusers.utils import load_image
from PIL import Image, ImageDraw, ImageFont
import gradio as gr
import numpy as np
import random
import os
import cv2

## Ouput Directory

In [ ]:
Config.OUTPUT_DIR = "/content/drive/MyDrive/output"
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)

## Load SDXL ControlNet Pipeline

In [ ]:
from diffusers import ControlNetModel, StableDiffusionXLControlNetPipeline
from transformers import pipeline

depth_estimator = pipeline('depth-estimation', model="LiheYoung/depth-anything-large-hf")

controlnets = {
    "Canny": ControlNetModel.from_pretrained("diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16).to("cuda"),
    "Depth": ControlNetModel.from_pretrained("diffusers/controlnet-depth-sdxl-1.0", torch_dtype=torch.float16).to("cuda")
}

base_model_id = "stabilityai/stable-diffusion-xl-base-1.0"

pipes = {
    "Canny": StableDiffusionXLControlNetPipeline.from_pretrained(
        base_model_id,
        controlnet=controlnets["Canny"],
        torch_dtype=torch.float16,
        variant="fp16"
    ).to("cuda"),

    "Depth": StableDiffusionXLControlNetPipeline.from_pretrained(
        base_model_id,
        controlnet=controlnets["Depth"],
        torch_dtype=torch.float16,
        variant="fp16"
    ).to("cuda"),

    "Both": StableDiffusionXLControlNetPipeline.from_pretrained(
        base_model_id,
        controlnet=[controlnets["Canny"], controlnets["Depth"]],
        torch_dtype=torch.float16,
        variant="fp16"
    ).to("cuda")
}

## Functions

In [ ]:
def resize(image, max_dim=1024):
    w, h = image.size
    scale = max_dim / max(w, h)
    return image.resize((int(w * scale), int(h * scale)))

def get_canny(image: Image.Image, low_thresh: int, high_thresh: int) -> Image.Image:
    image = resize(image)
    gray = np.array(image.convert("L"))
    edges = cv2.Canny(gray, low_thresh, high_thresh)
    edges_rgb = np.stack([edges]*3, axis=-1).astype(np.uint8)
    return Image.fromarray(edges_rgb)

def get_depth(image: Image.Image) -> Image.Image:
    image = resize(image)
    depth_image = depth_estimator(image)['depth']
    depth_np = np.array(depth_image)
    depth_np = (depth_np - depth_np.min()) / (depth_np.max() - depth_np.min())
    depth_np = (depth_np * 255).astype(np.uint8)
    depth_rgb = np.stack([depth_np] * 3, axis=2)
    return Image.fromarray(depth_rgb)

def get_combined(image: Image.Image, low_thresh: int, high_thresh: int) -> Image.Image:
    base_size = (1024, 1024)
    canny = np.array(get_canny(image, low_thresh, high_thresh).resize(base_size))
    depth = np.array(get_depth(image).resize(base_size))
    combined = np.maximum(canny, depth)
    return Image.fromarray(combined)

def save(prompt, steps, control_type, image):
    Config.PROMPT = prompt
    Config.STEPS = steps
    Config.AUTHOR = "James"
    Config.ALGO_TYPE = "Gradio ControlNet"
    Config.ALGO_NAME = "SDXL"

    set_image_path()
    save_image(image)

def generate_with_controlnet(
    image, control_type, prompt,
    guidance, steps,
    canny_low, canny_high,
    strength, controlnet_scale
):
    image_resized = image.resize((1024, 1024))

    if control_type == "Canny":
        control_image = get_canny(image, canny_low, canny_high).resize((1024, 1024)).convert("RGB")
        base_input = control_image.copy()

        out = pipes["Canny"](
            prompt=prompt,
            image=base_input,
            control_image=control_image,
            guidance_scale=guidance,
            num_inference_steps=steps,
            strength=strength,
            controlnet_conditioning_scale=float(controlnet_scale)
        ).images[0]

        save(prompt, steps, control_type, out)
        return image, control_image, out


    elif control_type == "Depth":
        control_image = get_depth(image).resize((1024, 1024)).convert("RGB")
        out = pipes["Depth"](
            prompt=prompt,
            image=image_resized,
            control_image=control_image,
            guidance_scale=guidance,
            num_inference_steps=steps,
            strength=strength,
            controlnet_conditioning_scale=float(controlnet_scale)
        ).images[0]

        save(prompt, steps, control_type, out)
        return image, control_image, out

    elif control_type == "Both":
        canny_control = get_canny(image, canny_low, canny_high).resize((1024, 1024)).convert("RGB")
        depth_control = get_depth(image).resize((1024, 1024)).convert("RGB")
        control_images = [canny_control, depth_control]
        base_input = canny_control.copy()
        image_list = [base_input] * 2

        out = pipes["Both"](
            prompt=prompt,
            image=image_list,
            control_image=control_images,
            guidance_scale=guidance,
            num_inference_steps=steps,
            strength=strength,
            controlnet_conditioning_scale=[float(controlnet_scale)] * 2
        ).images[0]

        save(prompt, steps, control_type, out)
        return image, canny_control, out

    else:
        raise ValueError(f"Unknown control type: {control_type}")

## Run Gradio

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# SDXL ControlNet")

    with gr.Row():
        img_input = gr.Image(label="Upload Image", type="pil")

        with gr.Column():
            control_type = gr.Dropdown(["Canny", "Depth", "Both"], value="Depth", label="Control Type")
            prompt = gr.Textbox(label="Prompt", value="A red house by the beach under a starry night sky", placeholder="Describe the scene")
            guidance = gr.Slider(0.0, 20.0, value=2.5, step=0.1, label="Guidance Scale")
            steps = gr.Slider(1, 50, value=25, step=1, label="Inference Steps")

            with gr.Accordion("Advanced Settings", open=False):
                canny_low = gr.Slider(0, 255, value=100, step=1, label="Canny Threshold 1 (Low)")
                canny_high = gr.Slider(0, 255, value=200, step=1, label="Canny Threshold 2 (High)")
                strength = gr.Slider(0.0, 1.0, value=0.75, step=0.01, label="Refinement Strength")
                controlnet_scale = gr.Slider(0.0, 2.0, value=0.8, step=0.1, label="ControlNet Conditioning Scale")

            run = gr.Button("Generate")

    with gr.Row():
        original = gr.Image(label="Input Image")
        control_map = gr.Image(label="Control Map Used")
        output = gr.Image(label="Generated Image")

    run.click(
        fn=generate_with_controlnet,
        inputs=[
            img_input, control_type, prompt,
            guidance, steps,
            canny_low, canny_high,
            strength, controlnet_scale
        ],
        outputs=[original, control_map, output]
    )

demo.launch(share=True, debug=True)